# Train the dual autoencoder on all slides

Trains the transcriptomics graph autoencoder and the histology-token autoencoder jointly on **all** slides, step by step.
There is no held-out slide here; leave-one-slide-out runs are done by `scripts/run_loso.py`, which reads the same configuration.

| Step | What happens | Code |
|---|---|---|
| Configuration | settings are set in one cell and written to `config/training_config.json` | `save_config` |
| 1. Load | transcriptomics, histology patches, nucleus coordinates and token embeddings of the selected slides | `load_slide_data` |
| 2. Prepare | gene selection, spatial graph, pairing, standardisation, slide-stratified loader | `build_pipeline` |
| 3. Model | dual autoencoder with the dimensions of the prepared data | `build_model` |
| 4. Pretrain | pretrain the token autoencoder | `pretrain_token_branch` |
| 5. Train | joint reconstruction and cell-to-token contrastive training | `train_joint` |
| 6. Evaluate | in-sample retrieval and slide leakage | `evaluate_embeddings` |
| 7. Save | checkpoint, embeddings, gene panel, configuration | `save_run` |

The functions are in `functions/training_pipeline.py`; the model, loss and loader are in `functions/dual_autoencoder.py`.

**Prerequisite:** every slide needs token embeddings from `scripts/token_preprocessing.py` (nuclei: `cell_token_embeddings.h5ad`,
bins: `bin_token_embeddings.h5ad`). Slides without them are skipped.

In [ ]:
import gc
import sys
from pathlib import Path

repo_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "functions").is_dir())
sys.path.insert(0, str(repo_root))

import pandas as pd
import torch

from functions.training_pipeline import (
    CONFIG_PATH, build_model, build_pipeline, data_paths, evaluate_embeddings, load_config,
    load_slide_data, pretrain_token_branch, save_config, save_run, train_joint,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## Configuration

The settings below are the configuration adopted after the ablation studies, for nuclei.
Running the next cell **overwrites** `config/training_config.json` with the values in the cell, so edit the values here.

Other scripts read the same file, for example a leave-one-slide-out run with `functions.training_pipeline.load_config()`.
To change a setting for those runs, edit the JSON directly. To run this notebook with the JSON instead, set `LOAD_FROM_JSON = True`; the file is then only read.

Key groups: `slides`/`exclude_slides` (None selects all slides), data location, model, spatial graph and batching,
contrastive loss, denoising and optimisation, token branch, gene selection.
`max_cells_per_slide` limits the per-epoch share of a very large slide and `oversample_slides` resamples slides to the given number of units per epoch.
`neighbor_radius` is only used for the patch metadata and does not affect the contrastive loss; `weight_reconstruction_by_patch_freq` switches the
inverse-density weighting of the reconstruction loss (the name is historical).

**Bins instead of nuclei:** set `bin_size="008um"`, `dir_cellseg="grid_bin"`, `token_filename="bin_token_embeddings.h5ad"`, `n_neighs=8`,
`spatial_graph_delaunay=False`, `spatial_graph_grid=True`, `max_cells_per_slide=610000` and `oversample_slides={"Normal_P3": 470000, "Normal_P5": 856000}` (these are `UNIT_PRESETS["bins"]` in `functions/training_pipeline.py`).

In [ ]:
LOAD_FROM_JSON = False

CONFIG = dict(
    slides=None,
    exclude_slides=[],
    output_name="all_slides",
    seed=0,

    bin_size="002um",
    dir_cellseg="cell_seg_stardist",
    token_filename="cell_token_embeddings.h5ad",
    gene_selection="svg",
    gene_panel=None,

    latent_dim=64,
    proj_dim=32,
    encoder="transconv",
    token_condition_on_slide=True,
    trans_condition_on_slide=True,

    n_neighs=6,
    spatial_graph_delaunay=True,
    spatial_graph_grid=False,
    num_neighbors=(10, 10),
    cells_per_batch=1024,
    max_cells_per_slide=300000,
    oversample_slides={"Normal_P3": 172000, "Normal_P5": 383000},
    neighbor_radius=2912,

    temperature=0.05,
    local_neighbor_radius=100,
    local_anchor_sim_threshold=None,
    local_anchor_sim_source="projected",
    local_positive_mode="exclude",
    weight_reconstruction_by_patch_freq=True,

    edge_dropout_p=0.1,
    gene_dropout_p=0.1,
    weight_decay=1e-5,
    n_epochs=100,
    lr=1e-3,
    warmup_epochs=5,

    skip_token_pretrain=False,
    freeze_token=False,
    token_pretrain_epochs=100,
    token_pretrain_lr=1e-3,
    token_pretrain_weight_decay=1e-5,
    token_pretrain_val_frac=0.1,
    token_pretrain_patience=15,
    token_lr_mult=0.3,
    max_local_contrast_weight=1.0,
    local_warmup_epochs=10,
)

if LOAD_FROM_JSON:
    CONFIG = load_config()
else:
    save_config(CONFIG)
print(f"Configuration {'read from' if LOAD_FROM_JSON else 'written to'} {CONFIG_PATH}")

## 1. Load the data

Reads only the selected slides from the merged transcriptomics file. Units must be identified by `Slide_ID` and `id`; the token table keeps only units with a histology match.

In [ ]:
data = load_slide_data(CONFIG)
print("Slides:", data["slides"])
print(f"{data['trans'].n_obs:,} units, {data['hist'].n_obs:,} histology patches, {data['token'].n_obs:,} token embeddings")

## 2. Prepare the training data

Selects genes (spatially variable genes by default; `gene_panel` uses an external panel instead), builds the spatial graph per slide,
keeps units that have a histology patch and a token embedding, standardises each modality and creates the slide-stratified neighbour loader.

The prepared data holds its own copy of the units, so the loaded tables are released afterwards.

In [ ]:
pipe = build_pipeline(data, CONFIG)
del data
gc.collect()

print(f"{pipe['n_svg']} genes, token dimension {pipe['token_dim']}, {pipe['n_slides']} slides")
print(pipe["trans_sub"].obs["Slide_ID"].value_counts().to_string())

## 3. Model

Transcriptomics graph encoder and decoder (conditioned on the slide), token encoder and decoder, and one projection head per modality for the contrastive loss.

In [ ]:
model = build_model(CONFIG, pipe, device)
print(f"{sum(p.numel() for p in model.parameters()):,} parameters")

## 4. Pretrain the token branch

The token autoencoder is trained on its own with a validation split and early stopping, and the best checkpoint is restored.
`skip_token_pretrain` disables this step.

In [ ]:
pretrain_info = pretrain_token_branch(model, pipe, CONFIG, device)
print(pretrain_info)

## 5. Joint training

Reconstruction of both modalities plus the contrastive loss between each unit's expression and token embedding.
The contrastive weight ramps up over `local_warmup_epochs`. If `local_frac_masked` approaches 1, semantic exclusion masks almost all negatives (collapse).

In [ ]:
history = train_joint(model, pipe, CONFIG, device)
pd.DataFrame(history).set_index("epoch")[["loss", "recon_trans", "recon_token", "contrast_local"]].plot(
    subplots=True, figsize=(7, 7), sharex=True
);

## 6. Evaluate

Cell-to-token retrieval (exact match and match of the independent cluster label) and how well the token embedding predicts the slide.
All slides were used for training, so these numbers are in-sample.

In [ ]:
evaluation = evaluate_embeddings(model, pipe, device)
evaluation["metrics"]

## 7. Save

Writes the checkpoint (weights, configuration, scalers, gene names, history), the embeddings with unit ids and slide names, the gene panel,
a copy of the configuration and the metrics into `data/combined/03_integration/autoencoders/<bin_size>/<output_name>/`.

In [ ]:
output_dir = save_run(data_paths(CONFIG)["output"], model, pipe, CONFIG, history, evaluation, pretrain_info)
print("Saved to", output_dir)